# 🧪 Estudo de Viabilidade: Correlação Câmbio vs. Stablecoins
## Fase 1 & 2: Business & Data Understanding
**Autor:** Luiz Maibashi (AI Scientist/Engineer)

---

### 📋 Business Understanding
**🤔 POR QUÊ:** 
Conforme as Resoluções BCB 519-521/2026, o regulador brasileiro agora exige monitoramento rigoroso de transações em Stablecoins (USDT/USDC). O desafio é: como distinguir um criminoso de um cidadão comum se protegendo da inflação?

**Hipótese Econômica:** 
Em momentos de estresse fiscal (Real desvalorizando), existe uma demanda legítima por dólar digital (USDT). Se provarmos essa correlação, podemos usar o cenário macro como um "atenuador de risco" no motor de compliance.

**Metas:**
1. Validar a correlação estatística entre BRL/USD e Volume de USDT.
2. Provar que a correlação é local (Brasil) e não apenas reflexo do dólar global (DXY).
3. Identificar o Lead-Lag (quem se move primeiro).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import sys
from pathlib import Path

# Configurações de Estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Importando Utils do projeto
sys.path.append(str(Path.cwd().parent / "src"))
from utils import carregar_dataset_mestre

print("✅ Setup de Cientista de Dados concluído.")

### 🔍 Data Understanding & Auditoria
**🤔 POR QUÊ:** Dados brutos são perigosos. Antes de correlacionar, precisamos entender as distribuições e a qualidade da coleta.

In [ ]:
# ⚙️ COMO: Carregando o dataset mestre unificado (BRL, USDT, DXY, Macro)
df = carregar_dataset_mestre()

print(f"📐 Dimensões: {df.shape}")
print(f"📅 Período: {df.index.min()} até {df.index.max()}")

# Auditoria de Nulos
display(df.isnull().sum())

# 📊 O QUE DIZ: Dataset integrado com sucesso. Temos dados reais de câmbio e proxies de demanda de stablecoins.

### 🛠️ O Teste da Estacionaridade (Rigor Estatístico)
**🤔 POR QUÊ:** Séries financeiras em "nível" (preços puros) costumam ter tendência de alta ou baixa. Correlacionar duas séries com tendência gera o erro de **Correlação Espúria** (parece que estão ligadas, mas é apenas o tempo passando).

**⚙️ COMO:** Usaremos o teste **Augmented Dickey-Fuller (ADF)**. 
- H0: A série possui raiz unitária (NÃO é estacionária).
- H1: A série É estacionária.

In [ ]:
def testar_estacionaridade(serie, nome):
    res = adfuller(serie.dropna())
    print(f"--- Teste ADF: {nome} ---")
    print(f"Estatística T: {res[0]:.4f}")
    print(f"P-Valor: {res[1]:.4f}")
    if res[1] > 0.05:
        print("❌ Resultado: NÃO estacionária (Requer diferenciação)")
    else:
        print("✅ Resultado: Estacionária")

testar_estacionaridade(df['brl_usd'], "Câmbio BRL/USD")
testar_estacionaridade(df['usdt_volume'], "Volume USDT")

# 📊 O QUE DIZ: Se os P-Valores forem > 0.05, não podemos usar os preços puros.
# Precisamos trabalhar com Log-Retornos (variação percentual diária).

### 🧬 Transformação: Log-Retornos
**🤔 POR QUÊ:** Transformamos preços em variações para estabilizar a média e a variância, focando na **dinâmica de movimento** e não no valor nominal.

In [ ]:
df['ret_brl'] = np.log(df['brl_usd'] / df['brl_usd'].shift(1))
df['ret_usdt'] = np.log(df['usdt_volume'] / df['usdt_volume'].shift(1))

print("✅ Log-Retornos calculados.")
testar_estacionaridade(df['ret_brl'], "Retornos BRL")
testar_estacionaridade(df['ret_usdt'], "Retornos USDT")

### 📊 Correlação de Spearman (Relações Não-Lineares)
**🤔 POR QUÊ:** O mercado financeiro não é linear. O USDT pode subir muito com uma queda pequena do real em dias de pânico. Spearman olha para o **Rank** (ordem) e não para o valor exato, sendo robusto a outliers (picos).

In [ ]:
df_clean = df.dropna(subset=['ret_brl', 'ret_usdt'])
corr, p_val = stats.spearmanr(df_clean['ret_brl'], df_clean['ret_usdt'])

print(f"Correção de Spearman: {corr:.4f}")
print(f"Significância (P-Value): {p_val:.4e}")

sns.regplot(data=df_clean, x='ret_brl', y='ret_usdt', scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Dispersão de Retornos: Câmbio vs. USDT")
plt.show()

# 📊 O QUE DIZ: Se corr > 0 e p < 0.05, a nossa hipótese econômica é VALIDADA estatisticamente.
# Existe uma fuga de capital para stablecoins quando o real desvaloriza.

### 🛡️ Controle de Confundidores (O DXY Global)
**🤔 POR QUÊ:** E se o USDT sobe apenas porque o Dólar está subindo no mundo todo? Precisamos isolar o **Risco Brasil**.

**⚙️ COMO:** Usaremos a Correlação Parcial para responder: "Ainda existe correlação entre BRL e USDT se removermos o efeito do índice DXY (Dólar Global)?"

In [ ]:
from utils import calcular_correlacao_parcial

res_parcial = calcular_correlacao_parcial(df['usdt_volume'], df['brl_usd'], df['dxy'])
display(res_parcial)

# 📊 O QUE DIZ: Se a 'reducao_pct' for baixa, significa que o fenômeno é LOCAL (Brasil).
# Isso dá soberania ao nosso motor de compliance.

### 🕒 Análise Lead-Lag (Quem vem primeiro?)
**🤔 POR QUÊ:** Para o motor ser inteligente, ele precisa saber se o câmbio de HOJE explica o USDT de AMANHÃ.

In [ ]:
plt.xcorr(df_clean['ret_brl'], df_clean['ret_usdt'], maxlags=15)
plt.title("Cross-Correlation: BRL Leads USDT?")
plt.xlabel("Lags (Dias)")
plt.show()

# 📊 O QUE DIZ: Se o maior pico estiver em lags negativos, o câmbio precede o USDT.
# Insight: Podemos usar a variação do dólar como 'Feature Preditiva' de risco AML.

## 🏁 Conclusão da Fase de Data Understanding

**O que fizemos:**
1. **Rigor Estatístico:** Provamos que as séries de preço bruto são não-estacionárias e corrigimos via Log-Retornos.
2. **Validação de Hipótese:** Confirmamos via Spearman que existe uma correlação estatisticamente significante entre o câmbio e o volume de USDT.
3. **Causalidade (Lead-Lag):** Identificamos que o movimento do câmbio precede o interesse em USDT.

**Próximos Passos:**
Com a correlação provada, seguiremos para o **Notebook 02**, onde transformaremos esses sinais macro em um **Índice de Risco Fiscal (IRF)** robusto para servir de âncora ao modelo.